# Chapter 5: Elliptic Curves - Examples

Jupyter notebook demonstrating the elliptic curve group law, point counts, and ECDH/ECDSA.

## 1. Point addition and doubling over a finite field

Implement the group law for $E: y^2 = x^3 + ax + b \pmod p$.

In [ ]:
def modinv(a, p):
    return pow(a, -1, p)

def ec_add(P, Q, a, p):
    """Add two points on E. None = point at infinity."""
    if P is None: return Q
    if Q is None: return P
    x1, y1 = P; x2, y2 = Q
    if x1 == x2 and (y1 + y2) % p == 0:
        return None  # P + (-P) = O
    if P == Q:
        lam = (3*x1*x1 + a) * modinv(2*y1, p) % p
    else:
        lam = (y2 - y1) * modinv(x2 - x1, p) % p
    x3 = (lam*lam - x1 - x2) % p
    y3 = (lam*(x1 - x3) - y1) % p
    return (x3, y3)

def ec_mul(k, P, a, p):
    """Scalar multiplication via double-and-add."""
    R = None
    while k:
        if k & 1:
            R = ec_add(R, P, a, p)
        P = ec_add(P, P, a, p)
        k >>= 1
    return R

# Curve from textbook: y^2 = x^3 + 2x + 2 over GF(17)
a, b, p = 2, 2, 17
P = (5, 1)
print("2P =", ec_add(P, P, a, p))
print("3P =", ec_add(ec_add(P, P, a, p), P, a, p))
print("17P =", ec_mul(17, P, a, p), "(should be infinity)")

## 2. List all points on E(F_p) and check Hasse's bound

For $E: y^2 = x^3 - x$ over $\mathbb{F}_7$, enumerate all points and verify $|q + 1 - \#E| \le 2\sqrt{q}$.

In [ ]:
import math

def points_on_curve(a, b, p):
    pts = []
    for x in range(p):
        rhs = (x**3 + a*x + b) % p
        # find y with y^2 = rhs
        for y in range(p):
            if (y*y) % p == rhs:
                pts.append((x, y))
    return pts

a, b, p = -1, 0, 7  # y^2 = x^3 - x over F_7
pts = points_on_curve(a % p, b, p)
print(f"Number of affine points: {len(pts)}")
print("Points:", pts)
total = len(pts) + 1  # + point at infinity
t = p + 1 - total
print(f"#E(F_7) = {total}")
print(f"Hasse: |q+1 - #E| = |{t}| <= {2*math.isqrt(7)}? {abs(t) <= 2*math.isqrt(7)}")

## 3. ECDH key exchange

Alice and Bob agree on a base point, exchange public keys (scalings of the base point), and recover the shared point $abP$.

In [ ]:
# Curve secp256k1-like small example over F_p
p = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F
a, b = 0, 7
G = (0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
     0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8)

# Note: full-size practical; just show the protocol struct
def ecdh(a_secret, b_secret):
    Q_a = ec_mul(a_secret, G, a, p)
    Q_b = ec_mul(b_secret, G, a, p)
    K_a = ec_mul(a_secret, Q_b, a, p)
    K_b = ec_mul(b_secret, Q_a, a, p)
    assert K_a == K_b
    return Q_a, Q_b, K_a

Qa, Qb, K = ecdh(12345, 54321)
print(f"Alice public: {Qa}")
print(f"Bob public:   {Qb}")
print(f"Shared key:   {K}")

## 4. ECDSA sign and verify (small example)

Tests the signature generation and verification steps.

In [ ]:
import hashlib, random

def ecdsa_sign(d, z, k, G, n, a, p):
    # R = k*G, r = x_R mod n, s = k^-1 (z + r*d)
    R = ec_mul(k, G, a, p)
    r = R[0] % n
    s = modinv(k, n) * (z + r*d) % n
    return r, s

def ecdsa_verify(Q, z, r, s, G, n, a, p):
    w = modinv(s, n)
    u1 = z*w % n
    u2 = r*w % n
    P1 = ec_mul(u1, G, a, p)
    P2 = ec_mul(u2, Q, a, p)
    v = ec_add(P1, P2, a, p)
    return v is not None and v[0] % n == r

# Use a tiny curve for demonstration
p = 67
a, b = 2, 3
G = (2, 22)  # some base point (order large enough for demo)
n = 79  # order
d = 27  # private key
Q = ec_mul(d, G, a, p)  # public key
z = 17  # hash of message
k = 11
r, s = ecdsa_sign(d, z, k, G, n, a, p)
print(f"Signature: (r={r}, s={s})")
print(f"Verify succeeds: {ecdsa_verify(Q, z, r, s, G, n, a, p)}")
print(f"Verify with wrong hash fails: {not ecdsa_verify(Q, z+1, r, s, G, n, a, p)}")